In [1]:
# Cell 1: Imports and Setup
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Import our custom modules
from model import SimpleVGG
from dataset import get_cifar10_dataloaders

# Setup
device = "cuda" if torch.cuda.is_available() else "cpu"
sns.set(style="whitegrid", font_scale=1.2)
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [2]:
device

'cuda'

In [3]:
# Cell 2: Load Model and Test Data
# IMPORTANT: We only need the test_loader here.
# We create a new set of data loaders but only use the test portion.
_, _, test_loader = get_cifar10_dataloaders(batch_size=128)

# Initialize the model structure
model = SimpleVGG(num_classes=10).to(device)

# Load the saved state dictionary from our best training epoch
model_path = 'best_cifar10_vgg.pth'
model.load_state_dict(torch.load(model_path))

# Crucially, set the model to evaluation mode
model.eval()
print("Model loaded successfully and set to evaluation mode.")

Full training set size: 50000
Training set size: 45000
Validation set size: 5000
Test set size: 10000


d:\Pytorch_DUMP\.venv\Lib\site-packages\torch\cuda\__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GeForce MX130 which is of cuda capability 5.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.5) - (12.0)
    
  warnings.warn(
d:\Pytorch_DUMP\.venv\Lib\site-packages\torch\cuda\__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
d:\Pytorch_DUMP\.venv\Lib\site-packages\torch\cuda\__init__.py:326: UserWarning: 
NVIDIA GeForce MX130 with CUDA capability sm_50 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the NVIDIA GeForce MX130 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


Model loaded successfully and set to evaluation mode.


In [4]:
# Cell 3: Final Unbiased Evaluation on Test Set
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Generate and print the classification report
print("--- Classification Report ---")
print(classification_report(all_labels, all_preds, target_names=classes))

# Plot the confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix on Test Set')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


RuntimeError: GET was unable to find an engine to execute this computation

In [ ]:
# Cell 4: Visualizing Sample Predictions (Successes and Failures)
def visualize_predictions(model, data_loader, num_images=20):
    model.eval()
    images, labels = next(iter(data_loader))
    images, labels = images.to(device), labels.to(device)
    
    with torch.no_grad():
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

    images = images.cpu().numpy()
    
    # Un-normalize for visualization
    mean = np.array([0.4914, 0.4822, 0.4465])
    std = np.array([0.2023, 0.1994, 0.2010])
    
    fig = plt.figure(figsize=(15, 12))
    for idx in np.arange(num_images):
        ax = fig.add_subplot(4, 5, idx+1, xticks=[], yticks=[])
        
        img = images[idx]
        img = np.transpose(img, (1, 2, 0)) # C, H, W -> H, W, C
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        ax.imshow(img)
        
        color = "green" if preds[idx] == labels[idx] else "red"
        ax.set_title(f"Pred: {classes[preds[idx]]}\n(True: {classes[labels[idx]]})", color=color)
    plt.tight_layout()
    plt.show()

visualize_predictions(model, test_loader)